In [ ]:
# --- BƯỚC 0: SETUP FOR RTX PRO 6000 BLACKWELL (KAGGLE OFFLINE) ---
import os
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")

OFFLINE_MODE = True
OFFLINE_DATASET_CANDIDATES = [
    "/kaggle/input/datasets/namthi/offline-kaggle-deps",
    "/kaggle/input/datasets/namthi/offline-kaggle-deps/offline_kaggle_deps",
    "/kaggle/input/offline-kaggle-deps",
    "/kaggle/input/offline-kaggle-deps/offline_kaggle_deps",
]

# torch 2.7.0+cu128 — hỗ trợ SM 12.0 (RTX PRO 6000 Blackwell)
TORCH_TARGET_VERSION = "2.7.0"

def _resolve_offline_dataset_root(candidates):
    for base in candidates:
        probe_dirs = [base, os.path.join(base, "offline_kaggle_deps"), os.path.join(base, "offline-kaggle-deps")]
        for d in probe_dirs:
            if not os.path.isdir(d):
                continue
            if os.path.isdir(os.path.join(d, "wheels")) and os.path.isfile(os.path.join(d, "requirements-runtime.txt")):
                return d
    return None

OFFLINE_DATASET_DIR = _resolve_offline_dataset_root(OFFLINE_DATASET_CANDIDATES)
OFFLINE_WHEEL_DIR = os.path.join(OFFLINE_DATASET_DIR, "wheels") if OFFLINE_DATASET_DIR else ""
OFFLINE_REQ_FILE = os.path.join(OFFLINE_DATASET_DIR, "requirements-runtime.txt") if OFFLINE_DATASET_DIR else ""

def _run(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

def _current_py_tag():
    return f"cp{sys.version_info.major}{sys.version_info.minor}"

def _wheel_matches_python_tag(filename):
    low = filename.lower()
    return ("py3-none-any" in low) or ("py3-none-manylinux" in low) or (_current_py_tag() in low)

def _find_wheel(package_name, version_prefix=None, require_python_match=False):
    if not os.path.isdir(OFFLINE_WHEEL_DIR):
        return None
    prefix = f"{package_name.lower()}-"
    cands = []
    for fn in os.listdir(OFFLINE_WHEEL_DIR):
        low = fn.lower()
        if not low.endswith(".whl"):
            continue
        if not low.startswith(prefix):
            continue
        if version_prefix and f"-{version_prefix}" not in low:
            continue
        if require_python_match and (not _wheel_matches_python_tag(fn)):
            continue
        cands.append(fn)
    if not cands:
        return None
    return os.path.join(OFFLINE_WHEEL_DIR, sorted(cands)[-1])

if OFFLINE_MODE and not OFFLINE_DATASET_DIR:
    raise FileNotFoundError(f"Offline dataset not found. Checked: {OFFLINE_DATASET_CANDIDATES}")

print(f">>> Using torch target: {TORCH_TARGET_VERSION}")
print(f">>> Runtime Python tag: {_current_py_tag()}")

torch_was_pinned = False

if OFFLINE_MODE:
    print(">>> OFFLINE MODE: install dependencies from dataset")
    print(f">>> OFFLINE_DATASET_DIR = {OFFLINE_DATASET_DIR}")

    if not os.path.isdir(OFFLINE_WHEEL_DIR):
        raise FileNotFoundError(f"Wheels folder not found: {OFFLINE_WHEEL_DIR}")
    if not os.path.isfile(OFFLINE_REQ_FILE):
        raise FileNotFoundError(f"requirements-runtime.txt not found: {OFFLINE_REQ_FILE}")

    # ── 1. Filter requirements: skip packages that should use Kaggle's preinstalled ──
    filtered_req = "/kaggle/working/requirements-runtime-filtered.txt"
    # QUAN TRỌNG: skip numpy/scipy để tránh "numpy.dtype size changed" error
    skip_prefixes = ("pyarrow", "torch", "torchvision", "torchaudio", "triton", "numpy", "scipy")
    with open(OFFLINE_REQ_FILE, "r", encoding="utf-8") as rf, open(filtered_req, "w", encoding="utf-8") as wf:
        for line in rf:
            s = line.strip().lower()
            if (not s) or s.startswith("#"):
                wf.write(line)
                continue
            normalized = s.replace(" ", "")
            if normalized.startswith(skip_prefixes):
                continue
            wf.write(line)

    # ── 2. Install non-torch deps (--no-deps tránh pip resolver conflict) ──
    print(f">>> Installing non-torch deps (filtered, --no-deps)...")
    _run([
        sys.executable, "-m", "pip", "install",
        "--no-index", "--find-links", OFFLINE_WHEEL_DIR,
        "--no-deps",
        "-r", filtered_req,
    ])

    # ── 3. Install nvidia-cusparselt TRƯỚC torch (provides libcusparseLt.so.0) ──
    cusparselt_whls = [
        f for f in os.listdir(OFFLINE_WHEEL_DIR)
        if f.endswith(".whl") and "cusparselt" in f.lower()
    ]
    for whl in cusparselt_whls:
        whl_path = os.path.join(OFFLINE_WHEEL_DIR, whl)
        print(f">>> Installing NVIDIA lib: {whl}")
        _run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", whl_path])

    # ── 4. Install torch cu128 (SM 12.0 support) ──
    torch_whl = _find_wheel("torch", TORCH_TARGET_VERSION, require_python_match=True)
    if not torch_whl:
        available = [f for f in os.listdir(OFFLINE_WHEEL_DIR) if f.lower().startswith("torch-") and f.endswith(".whl")]
        print(f"WARN: No torch {TORCH_TARGET_VERSION} wheel found. Available: {available}")
        print("WARN: Keep using runtime torch preinstalled by Kaggle.")
    else:
        print(f">>> Installing torch: {os.path.basename(torch_whl)}")
        _run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", torch_whl])
        torch_was_pinned = True

    # ── 5. Install torchvision/torchaudio cu128 ──
    tv_whl = _find_wheel("torchvision", require_python_match=True)
    ta_whl = _find_wheel("torchaudio", require_python_match=True)
    if tv_whl:
        print(f">>> Installing torchvision: {os.path.basename(tv_whl)}")
        _run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", tv_whl])
    if ta_whl:
        print(f">>> Installing torchaudio: {os.path.basename(ta_whl)}")
        _run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", ta_whl])

    # ── 6. Install colpali-engine ──
    colpali_whls = [f for f in os.listdir(OFFLINE_WHEEL_DIR) if f.endswith(".whl") and "colpali" in f.lower()]
    if colpali_whls:
        colpali_path = os.path.join(OFFLINE_WHEEL_DIR, sorted(colpali_whls)[0])
        print(f">>> Installing colpali-engine (--no-deps): {os.path.basename(colpali_path)}")
        _run([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", OFFLINE_WHEEL_DIR, "--no-deps", colpali_path])
    else:
        raise FileNotFoundError("No colpali-engine wheel found.")

else:
    print(">>> ONLINE MODE")
    _run([sys.executable, "-m", "pip", "uninstall", "-y", "tensorflow", "pyarrow"])
    _run([sys.executable, "-m", "pip", "install", "pyarrow<20.0.0"])
    _run([sys.executable, "-m", "pip", "install", "-U", "git+https://github.com/illuin-tech/colpali"])
    _run([sys.executable, "-m", "pip", "install", "-U", f"torch=={TORCH_TARGET_VERSION}", "transformers", "accelerate", "peft", "bitsandbytes"])
    _run([sys.executable, "-m", "pip", "install", "-U", "torchvision", "torchaudio"])

# ── Preload NVIDIA libs (torch cu128 cần libcusparseLt.so.0) ──
# nvidia pip packages đặt .so files trong path riêng, torch không tự tìm được.
# Phải dùng ctypes preload trước khi import torch.
import ctypes
import glob

nvidia_lib_dirs = glob.glob("/usr/local/lib/python*/dist-packages/nvidia/*/lib")
if nvidia_lib_dirs:
    existing_ld = os.environ.get("LD_LIBRARY_PATH", "")
    os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_lib_dirs) + (":" + existing_ld if existing_ld else "")
    print(f">>> Added {len(nvidia_lib_dirs)} NVIDIA lib dirs to LD_LIBRARY_PATH")

# Preload specific libraries that torch needs
for pattern in [
    "/usr/local/lib/python*/dist-packages/nvidia/cusparselt/lib/libcusparseLt.so*",
    "/usr/local/lib/python*/dist-packages/nvidia/*/lib/lib*.so*",
]:
    for lib_path in sorted(glob.glob(pattern)):
        try:
            ctypes.CDLL(lib_path, mode=ctypes.RTLD_GLOBAL)
        except Exception:
            pass  # Không phải tất cả .so đều cần preload

cusparselt_found = glob.glob("/usr/local/lib/python*/dist-packages/nvidia/cusparselt/lib/libcusparseLt.so*")
if cusparselt_found:
    print(f">>> Preloaded libcusparseLt: {cusparselt_found[0]}")
else:
    print("WARN: libcusparseLt.so not found in nvidia packages!")

import torch

print(f">>> torch version: {torch.__version__}")
if torch_was_pinned and (not torch.__version__.startswith(TORCH_TARGET_VERSION)):
    raise RuntimeError(
        f"Expected torch {TORCH_TARGET_VERSION}, but got {torch.__version__}. "
        "Please verify offline wheel pack and restart kernel."
    )
if not torch_was_pinned:
    print(">>> Using runtime torch (wheel pin skipped).")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    gpu_name = props.name
    vram_gb = props.total_memory / (1024 ** 3)
    cc_major, cc_minor = torch.cuda.get_device_capability(0)
    print(f">>> GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB | CC={cc_major}.{cc_minor}")
else:
    raise RuntimeError("CUDA is not available.")

PERF_CFG = {
    "GPU_NAME": gpu_name,
    "VRAM_GB": round(vram_gb, 1),
    "BATCH_SIZE_ENCODE": 12,
    "BATCH_Q_EVAL": 12,
    "DOC_CHUNK_SIZE": 1024,
    "SAVE_EVERY": 900,
}
print(
    ">>> RTX6000 config: "
    f"BATCH_SIZE_ENCODE={PERF_CFG['BATCH_SIZE_ENCODE']}, "
    f"BATCH_Q_EVAL={PERF_CFG['BATCH_Q_EVAL']}, "
    f"DOC_CHUNK_SIZE={PERF_CFG['DOC_CHUNK_SIZE']}"
)

print(">>> Setup dependencies done")


In [ ]:
# --- PATCH: Fix colpali_engine import conflict (stub ALL submodules except idefics3) ---
import types, sys

def _patch_colpali_stub_all():
    # colpali-engine 0.3.14 imports nhiều model families cần transformers >= 5.x
    # Ta chỉ dùng idefics3 → stub hết các model khác.
    problem_modules = [
        # gemma3 (cần Gemma3Model — không có trong transformers 4.51.x)
        "colpali_engine.models.gemma3",
        "colpali_engine.models.gemma3.bigemma3",
        "colpali_engine.models.gemma3.colgemma3",
        # modernvbert
        "colpali_engine.models.modernvbert",
        "colpali_engine.models.modernvbert.bivbert",
        "colpali_engine.models.modernvbert.colvbert",
        # paligemma (stub cho an toàn)
        "colpali_engine.models.paligemma",
        "colpali_engine.models.paligemma.bipali",
        "colpali_engine.models.paligemma.colpali",
        "colpali_engine.models.paligemma.bipali_proj",
        # qwen2
        "colpali_engine.models.qwen2",
        "colpali_engine.models.qwen2.biqwen2",
        "colpali_engine.models.qwen2.colqwen2",
        # qwen3
        "colpali_engine.models.qwen3",
        "colpali_engine.models.qwen3.biqwen3",
        "colpali_engine.models.qwen3.colqwen3",
        # qwen3_5
        "colpali_engine.models.qwen3_5",
        "colpali_engine.models.qwen3_5.biqwen3_5",
        "colpali_engine.models.qwen3_5.colqwen3_5",
        # qwen_omni
        "colpali_engine.models.qwen_omni",
        "colpali_engine.models.qwen_omni.colqwen2_5_omni",
    ]

    stub_class_map = {
        "colpali_engine.models.gemma3": [
            "BiGemma3", "BiGemmaProcessor3",
            "ColGemma3", "ColGemmaProcessor3",
        ],
        "colpali_engine.models.modernvbert": [
            "BiModernVBert", "BiModernVBertProcessor",
            "ColModernVBert", "ColModernVBertProcessor",
        ],
        "colpali_engine.models.paligemma": [
            "BiPali", "BiPaliProcessor", "BiPaliProj",
            "ColPali", "ColPaliProcessor",
        ],
        "colpali_engine.models.qwen2": [
            "BiQwen2", "BiQwen2Processor",
            "ColQwen2", "ColQwen2Processor",
        ],
        "colpali_engine.models.qwen3": [
            "BiQwen3", "BiQwen3Processor",
            "ColQwen3", "ColQwen3Processor",
        ],
        "colpali_engine.models.qwen3_5": [
            "BiQwen3_5", "BiQwen3_5Processor",
            "ColQwen3_5", "ColQwen3_5Processor",
        ],
        "colpali_engine.models.qwen_omni": [
            "ColQwen2_5Omni", "ColQwen2_5OmniProcessor",
        ],
    }

    for name in problem_modules:
        stub = types.ModuleType(name)
        sys.modules[name] = stub

    for mod_name, cls_list in stub_class_map.items():
        mod = sys.modules[mod_name]
        for cls_name in cls_list:
            setattr(mod, cls_name, type(cls_name, (), {}))

    # Xóa cache colpali_engine để force re-import sạch
    stale = [k for k in sys.modules
             if k.startswith("colpali_engine") and k not in problem_modules]
    for k in stale:
        del sys.modules[k]

    print(f">>> Stubbed {len(problem_modules)} problematic submodules")

_patch_colpali_stub_all()

# Verify
from colpali_engine.models.idefics3 import ColIdefics3, ColIdefics3Processor
print(f"✅ ColIdefics3 ready: {ColIdefics3}")


In [ ]:
# --- BƯỚC 1: SETUP DATA (PAGE-LEVEL BATCH MODE) ---
import glob
import os
import json
import io
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm

# Chỉnh mốc batch tại đây
START_IDX = 1
END_IDX = 2

SEARCH_ROOT = "/kaggle/input"
ANNOTATIONS_PATH = "/kaggle/input/mmdocir-eval-data/MMDocIR_annotations.jsonl"
PARQUET_PATH = "/kaggle/input/mmdocir-eval-data/MMDocIR_layouts.parquet"
ENHANCED_JSONL_DIR = None
ENHANCED_IMG_DIR = None

for root, dirs, files in os.walk(SEARCH_ROOT):
    if "LAYOUT_CONTENT_FINAL" in root:
        ENHANCED_JSONL_DIR = root
    if "IMAGE ENHACED" in root or sum(1 for f in files if f.endswith(".jpg")) > 1000:
        ENHANCED_IMG_DIR = root

if not ENHANCED_JSONL_DIR:
    ENHANCED_JSONL_DIR = "/kaggle/input/siglip-qwen-enhaced/SIGLIP_QWEN_ENHACED/LAYOUT_CONTENT_FINAL"
if not ENHANCED_IMG_DIR:
    ENHANCED_IMG_DIR = "/kaggle/input/siglip-qwen-enhaced/SIGLIP_QWEN_ENHACED/IMAGE ENHACED"

print(f"Enhanced JSONL dir: {ENHANCED_JSONL_DIR}")
print(f"Enhanced IMG dir:   {ENHANCED_IMG_DIR}")

# A. Doc có câu hỏi trong annotations
valid_docs_in_qa = set()
with open(ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        try:
            d = json.loads(line)
            valid_docs_in_qa.add(d["doc_name"].replace(".pdf", ""))
        except Exception:
            pass

# B. Doc có file jsonl enrichment
available_jsonls = glob.glob(os.path.join(ENHANCED_JSONL_DIR, "*.jsonl"))
jsonl_map = {}
for p in available_jsonls:
    fname = os.path.basename(p).replace("_layout.jsonl", "")
    jsonl_map[fname] = p

intersection_docs = sorted(list(valid_docs_in_qa.intersection(jsonl_map.keys())))
if not intersection_docs:
    raise ValueError("Không tìm thấy tài liệu chung giữa annotations và enrichment JSONL")

target_doc_names = intersection_docs[START_IDX:END_IDX]
target_files = [jsonl_map[d] for d in target_doc_names]

print(f"PROCESSING BATCH [{START_IDX}:{END_IDX}]")
print(f"Selected docs ({len(target_doc_names)}): {target_doc_names}")
if len(target_doc_names) == 0:
    raise ValueError("Batch rỗng, kiểm tra START_IDX/END_IDX")

print("Loading backbone parquet...")
df_orig = pd.read_parquet(PARQUET_PATH)
df_orig["join_doc_name"] = df_orig["doc_name"].str.replace(".pdf", "", regex=False)
df_orig = df_orig[df_orig["join_doc_name"].isin(target_doc_names)]

print("Loading enrichment jsonl...")
dfs = []
for f in tqdm(target_files, desc="Reading JSONLs"):
    try:
        temp = pd.read_json(f, lines=True)
        temp["join_doc_name"] = os.path.basename(f).replace("_layout.jsonl", "")
        if "layout" in temp.columns:
            temp = temp.rename(columns={"layout": "layout_id"})

        cols = ["join_doc_name", "layout_id", "vlm_text", "img_enhanced_path"]
        if "text_level" in temp.columns:
            cols.append("text_level")
        temp = temp[[c for c in cols if c in temp.columns]]
        dfs.append(temp)
    except Exception:
        pass

if dfs:
    df_enh = pd.concat(dfs, ignore_index=True)
    df_enh = df_enh.rename(columns={"vlm_text": "vlm_text_enhanced", "text_level": "text_level_enhanced"})
else:
    df_enh = pd.DataFrame()

print("Merging and contextualizing...")
df_final = pd.merge(df_orig, df_enh, on=["join_doc_name", "layout_id"], how="left")
df_final = df_final.sort_values(by=["join_doc_name", "page_id", "layout_id"])

def identify_header(row):
    if row.get("type") in ["title", "section_header", "header"]:
        return str(row.get("text", ""))
    if pd.notna(row.get("text_level_enhanced")):
        return str(row.get("text", ""))
    return np.nan

df_final["temp_header"] = df_final.apply(identify_header, axis=1)
df_final["current_section"] = df_final.groupby("join_doc_name")["temp_header"].ffill().fillna("General Content")

enh_image_map = {}
if os.path.exists(ENHANCED_IMG_DIR):
    for f in glob.glob(os.path.join(ENHANCED_IMG_DIR, "*")):
        enh_image_map[os.path.basename(f)] = f

def get_best_sources(row):
    img_type, img_data = None, None
    if pd.notna(row.get("img_enhanced_path")):
        fname = os.path.basename(str(row["img_enhanced_path"]))
        if fname in enh_image_map:
            img_type, img_data = "path", enh_image_map[fname]
    if img_data is None and pd.notna(row.get("image_binary")):
        img_type, img_data = "binary", row["image_binary"]

    raw_content = ""
    if pd.notna(row.get("vlm_text_enhanced")) and len(str(row["vlm_text_enhanced"])) > 5:
        raw_content = str(row["vlm_text_enhanced"])
    elif pd.notna(row.get("text")) and len(str(row["text"])) > 5:
        raw_content = str(row["text"])
    elif pd.notna(row.get("ocr_text")) and len(str(row["ocr_text"])) > 5:
        raw_content = str(row["ocr_text"])
    elif pd.notna(row.get("vlm_text")):
        raw_content = str(row["vlm_text"])

    section = row.get("current_section", "")
    final_text_prompt = f"Section: {section}\nContent: {raw_content}"
    if len(final_text_prompt) < 10:
        final_text_prompt = "Document layout."

    return pd.Series([img_type, img_data, final_text_prompt], index=["img_type", "img_data", "final_text"])

processed = df_final.apply(get_best_sources, axis=1)
sample_layouts_df = pd.concat([df_final, processed], axis=1).dropna(subset=["img_type"]).reset_index(drop=True)

# ---------------- PAGE-LEVEL BUILD ----------------
# Một page sẽ có 1 ảnh đại diện + text gom từ tất cả layout thuộc page đó.
if "page_idx" in sample_layouts_df.columns:
    page_col = "page_idx"
else:
    page_col = "page_id"

sample_layouts_df["safe_page"] = pd.to_numeric(sample_layouts_df[page_col], errors="coerce").fillna(-999).astype(int)
sample_layouts_df = sample_layouts_df[sample_layouts_df["safe_page"] >= 0].copy()

for c in ["img_type", "img_data", "final_text", "current_section"]:
    if c not in sample_layouts_df.columns:
        sample_layouts_df[c] = np.nan

# Khóa page dùng xuyên suốt index + eval.
sample_layouts_df["page_key"] = sample_layouts_df.apply(
    lambda r: f"{r['join_doc_name']}__p{int(r['safe_page'])}", axis=1
)

page_rows = []
page_to_layout_indices = {}

group_cols = ["join_doc_name", "safe_page"]
for (doc_name, safe_page), grp in sample_layouts_df.groupby(group_cols, sort=True):
    grp = grp.sort_values(by=["layout_id"]).copy()
    page_key = f"{doc_name}__p{int(safe_page)}"

    # 1) Ảnh page đại diện: ưu tiên path rồi đến binary.
    rep = grp[grp["img_type"].isin(["path", "binary"])].head(1)
    if rep.empty:
        continue
    rep_row = rep.iloc[0]

    # 2) Text page: concat unique layout text theo thứ tự layout_id.
    seen = set()
    text_chunks = []
    for _, r in grp.iterrows():
        t = str(r.get("final_text", "")).strip()
        if not t or t in seen:
            continue
        seen.add(t)
        text_chunks.append(t)

    page_text = "\n\n".join(text_chunks)
    if len(page_text) < 10:
        page_text = "Document page."

    page_rows.append(
        {
            "join_doc_name": doc_name,
            "safe_page": int(safe_page),
            "page_key": page_key,
            "img_type": rep_row["img_type"],
            "img_data": rep_row["img_data"],
            "final_text": page_text[:8000],
            "n_layouts": int(len(grp)),
        }
    )
    page_to_layout_indices[page_key] = grp.index.astype(int).tolist()

sample_pages_df = pd.DataFrame(page_rows).sort_values(by=["join_doc_name", "safe_page"]).reset_index(drop=True)
if sample_pages_df.empty:
    raise ValueError("Không tạo được sample_pages_df. Kiểm tra dữ liệu page/layout.")

page_key_to_index = {k: i for i, k in enumerate(sample_pages_df["page_key"].tolist())}

globals()["sample_layouts_df"] = sample_layouts_df
globals()["sample_pages_df"] = sample_pages_df
globals()["page_to_layout_indices"] = page_to_layout_indices
globals()["page_key_to_index"] = page_key_to_index

doc_page_counts = sample_pages_df.groupby("join_doc_name").size().to_dict()

print("=" * 60)
print(f"BATCH READY (PAGE-LEVEL): [{START_IDX}:{END_IDX}]")
print(f"Docs: {target_doc_names}")
print(f"Total layouts kept: {len(sample_layouts_df)}")
print(f"Total pages indexed: {len(sample_pages_df)}")
print(f"Pages per doc: {doc_page_counts}")
print("=" * 60)

In [ ]:
# --- BƯỚC 2: LOAD MODEL ---
print(">>> BƯỚC 2: Loading ColSmolVLM-500M...")

import torch
import gc
import os
import sys
import types
import unittest.mock as mock
import transformers

gc.collect()
torch.cuda.empty_cache()

print(f">>> transformers version: {transformers.__version__}")

# Stub bitsandbytes
import importlib
import importlib.util

_bnb_stub = types.ModuleType("bitsandbytes")
_bnb_stub.__spec__ = None
_bnb_stub.__version__ = "0.0.0"
sys.modules["bitsandbytes"] = _bnb_stub

import importlib.util as _ilu
_orig_find_spec = _ilu.find_spec

def _patched_find_spec(name, package=None):
    if name == "bitsandbytes":
        return None
    return _orig_find_spec(name, package)

_ilu.find_spec = _patched_find_spec
print(">>> bitsandbytes stubbed + find_spec patched OK")

def _import_colsmol_classes():
    from colpali_engine.models.idefics3 import ColIdefics3, ColIdefics3Processor
    return ColIdefics3, ColIdefics3Processor

try:
    ColIdefics3, ColIdefics3Processor = _import_colsmol_classes()
except Exception as e:
    msg = str(e)
    if "conversion_mapping" in msg or "get_device" in msg:
        raise RuntimeError(
            "Xung đột version transformers/accelerate. "
            "Hãy Restart Kernel, chạy lại BƯỚC 0, rồi BƯỚC 2."
        ) from e
    raise

device = "cuda" if torch.cuda.is_available() else "cpu"

# ── FIX BLACKWELL ────────────────────────────────────────────────────────
# RTX PRO 6000 = Blackwell (SM10.0). Torch 2.6.0 chưa có native bf16/fp16
# kernel cho SM100 → dùng float32 storage, autocast trong encode.
if device == "cuda":
    major, minor = torch.cuda.get_device_capability(0)
    gpu_name = torch.cuda.get_device_name(0)
    IS_BLACKWELL = (major >= 10)   # Blackwell = SM100+
    if IS_BLACKWELL:
        RUNTIME_DTYPE  = torch.bfloat16   # torch 2.7+cu128 hỗ trợ native bfloat16 cho SM12.0
        AUTOCAST_DTYPE = torch.bfloat16
        print(f">>> GPU: {gpu_name} | CC={major}.{minor} | *** BLACKWELL DETECTED ***")
        print(">>> Strategy: bfloat16 native (torch 2.7+cu128 supports SM12.0)")

    elif major >= 8:
        RUNTIME_DTYPE  = torch.bfloat16
        AUTOCAST_DTYPE = torch.bfloat16
        print(f">>> GPU: {gpu_name} | CC={major}.{minor} | Runtime dtype=bfloat16")
    else:
        RUNTIME_DTYPE  = torch.float16
        AUTOCAST_DTYPE = torch.float16
        print(f">>> GPU: {gpu_name} | CC={major}.{minor} | Runtime dtype=float16")
else:
    IS_BLACKWELL   = False
    RUNTIME_DTYPE  = torch.float32
    AUTOCAST_DTYPE = torch.float32

globals()['IS_BLACKWELL']   = IS_BLACKWELL
globals()['AUTOCAST_DTYPE'] = AUTOCAST_DTYPE
# ─────────────────────────────────────────────────────────────────────────

if globals().get("OFFLINE_MODE", False):
    os.environ["HF_HUB_OFFLINE"]      = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"

def _first_existing_path(candidates):
    for p in candidates:
        if os.path.isdir(p):
            return p
    return None

BASE_MODEL_CANDIDATES = [
    "/kaggle/input/datasets/namthi/colsmolvlm-instruct-500m-base",
]
ADAPTER_DIR = "/kaggle/input/datasets/namthi/colsmol-500m"

base_dir = _first_existing_path(BASE_MODEL_CANDIDATES)
if not base_dir:
    raise FileNotFoundError("Base model not found. Kiem tra lai path dataset.")

print(f">>> Base model: {base_dir}")
print(f">>> Adapter:    {ADAPTER_DIR}")

model_load_kwargs = {
    "torch_dtype":              RUNTIME_DTYPE,
    "device_map":         "cpu",
    "attn_implementation": "eager",
    "local_files_only":   True,
}
print(">>> Loading base model (GPU)...")
model_load_kwargs = {
    "torch_dtype":         RUNTIME_DTYPE,
    "device_map":          "auto",          # Load thẳng lên GPU (95GB VRAM!)
    "attn_implementation": "eager",
    "local_files_only":    True,
}
model = ColIdefics3.from_pretrained(base_dir, **model_load_kwargs)
print(">>> Base model loaded OK")

print(">>> Loading LoRA adapter...")
from peft import PeftModel, PeftConfig

peft_config = PeftConfig.from_pretrained(ADAPTER_DIR, local_files_only=True)
if hasattr(peft_config, "load_in_8bit"):  peft_config.load_in_8bit = False
if hasattr(peft_config, "load_in_4bit"):  peft_config.load_in_4bit = False
print(">>> PeftConfig loaded OK")

model = PeftModel.from_pretrained(
    model, ADAPTER_DIR,
    config=peft_config,
    local_files_only=True,
    is_trainable=False,
)
print(">>> PeftModel loaded OK, merging...")

model = model.merge_and_unload()
print(">>> Merge done")

model = model.eval()

globals()["MODEL_RUNTIME_DTYPE"] = RUNTIME_DTYPE

processor_load_kwargs = {"local_files_only": True}
processor = ColIdefics3Processor.from_pretrained(base_dir, **processor_load_kwargs)
print(">>> Model & Processor Ready!")


In [ ]:
# --- BƯỚC 3: FUSED INDEXING (PAGE-LEVEL) ---
print(">>> BƯỚC 3: Creating Fused Vectors (Page-Level)")

import io
import os
import gc
import pickle
import torch
from PIL import Image
from tqdm.notebook import tqdm

WORKING_DIR = "/kaggle/working"
INDEX_SUFFIX = f"{START_IDX}_{END_IDX}" if "START_IDX" in globals() and "END_IDX" in globals() else "full"
FINAL_INDEX_PATH = os.path.join(WORKING_DIR, f"colsmol_fused_index_{INDEX_SUFFIX}.pkl")
CHECKPOINT_PATH = os.path.join(WORKING_DIR, f"colsmol_checkpoint_{INDEX_SUFFIX}.pkl")

BATCH_SIZE = int(globals().get("PERF_CFG", {}).get("BATCH_SIZE_ENCODE", 12))
SAVE_EVERY = int(globals().get("PERF_CFG", {}).get("SAVE_EVERY", 900))
print(f"Using BATCH_SIZE={BATCH_SIZE}, SAVE_EVERY={SAVE_EVERY}")

fused_index = []
start_idx = 0
skip_encoding = False
cpu_fallback_activated = False

if os.path.exists(FINAL_INDEX_PATH):
    print(f" Found completed index: {FINAL_INDEX_PATH}")
    with open(FINAL_INDEX_PATH, "rb") as f:
        loaded = pickle.load(f)
    fused_index = loaded["fused_index"] if isinstance(loaded, dict) and "fused_index" in loaded else loaded
    if len(fused_index) > 0:
        print(f" Loaded {len(fused_index)} pages. Skip encoding.")
        skip_encoding = True
    else:
        print(" Existing index is empty. Rebuild from scratch.")
elif os.path.exists(CHECKPOINT_PATH):
    print(" Found checkpoint. Resuming...")
    try:
        with open(CHECKPOINT_PATH, "rb") as f:
            ckpt = pickle.load(f)
        if isinstance(ckpt, dict):
            fused_index = ckpt.get("fused_index", [])
            start_idx = int(ckpt.get("next_row", len(fused_index)))
        else:
            fused_index = ckpt
            start_idx = len(fused_index)
        print(f" Restored {len(fused_index)} pages. Continue from row {start_idx}.")
    except Exception as e:
        print(f" Checkpoint corrupted ({e}). Start from scratch.")
        fused_index, start_idx = [], 0
else:
    print(" Fresh run.")

if not skip_encoding:
    if "sample_pages_df" not in globals():
        raise ValueError("sample_pages_df not found. Run BƯỚC 1 first.")

    total_pages = len(sample_pages_df)
    print(f" Total pages to process: {total_pages}")

    if start_idx >= total_pages and len(fused_index) == 0:
        start_idx = 0

    remaining_df = sample_pages_df.iloc[start_idx:].reset_index(drop=True)

    def load_img_safe(row):
        try:
            img = None
            if row["img_type"] == "path":
                img = Image.open(row["img_data"])
            elif row["img_type"] == "binary":
                img = Image.open(io.BytesIO(row["img_data"]))
            if img is not None:
                img = img.convert("RGB")
                if img.width < 14 or img.height < 14:
                    return Image.new("RGB", (224, 224), "white")
                return img
        except Exception:
            pass
        return Image.new("RGB", (224, 224), "white")

    _autocast_dtype = globals().get("AUTOCAST_DTYPE", torch.bfloat16)
    _use_autocast = device == "cuda"

    def encode_batch(batch_imgs, batch_texts, run_device):
        ctx = torch.autocast(
            device_type="cuda",
            dtype=_autocast_dtype,
            enabled=_use_autocast,
        )
        with ctx:
            inputs_vis = processor.process_images(batch_imgs).to(run_device)
            out_vis = model(**inputs_vis)
            emb_vis_list = list(torch.unbind(out_vis.float().to("cpu")))

            inputs_txt = processor.process_queries(batch_texts, max_length=512, suffix="").to(run_device)
            out_txt = model(**inputs_txt)
            emb_txt_list = list(torch.unbind(out_txt.float().to("cpu")))

        for k in range(len(emb_vis_list)):
            fused_vec = torch.cat([emb_vis_list[k], emb_txt_list[k]], dim=0).to(torch.float16).contiguous()
            fused_index.append(fused_vec)

    batch_imgs, batch_texts = [], []

    for i, row in tqdm(remaining_df.iterrows(), total=len(remaining_df), desc="Encoding pages"):
        current_real_idx = start_idx + i

        img = load_img_safe(row)
        txt = str(row["final_text"]).replace("<image>", " ")[:4000]

        batch_imgs.append(img)
        batch_texts.append(txt)

        if len(batch_imgs) >= BATCH_SIZE or i == len(remaining_df) - 1:
            with torch.no_grad():
                try:
                    encode_batch(batch_imgs, batch_texts, device)
                except Exception as e:
                    err_msg = str(e).lower()
                    is_cuda_err = (
                        "no kernel image is available for execution on the device" in err_msg
                        or "cuda error" in err_msg
                    )
                    if is_cuda_err and not cpu_fallback_activated:
                        print(f" CUDA error even with autocast: {e}")
                        print(" Switching to CPU fallback for all remaining batches...")
                        try:
                            model = model.to(device="cpu", dtype=torch.float32)
                        except Exception:
                            model = model.to("cpu")
                        device = "cpu"
                        globals()["device"] = "cpu"
                        globals()["_use_autocast"] = False
                        _use_autocast = False
                        cpu_fallback_activated = True
                        torch.cuda.empty_cache()
                        gc.collect()
                        try:
                            encode_batch(batch_imgs, batch_texts, "cpu")
                        except Exception as e2:
                            print(f" CPU fallback also failed: {e2}. Skip batch.")
                    else:
                        print(f" Batch {current_real_idx} error: {e}. Skip batch.")

            batch_imgs, batch_texts = [], []

            if current_real_idx % 100 == 0:
                torch.cuda.empty_cache()
                gc.collect()

            if len(fused_index) > 0 and len(fused_index) % SAVE_EVERY == 0:
                with open(CHECKPOINT_PATH, "wb") as f:
                    pickle.dump({"fused_index": fused_index, "next_row": current_real_idx + 1}, f)
                print(f" Checkpoint saved: {len(fused_index)} pages")

    with open(FINAL_INDEX_PATH, "wb") as f:
        pickle.dump(fused_index, f)

    if os.path.exists(CHECKPOINT_PATH):
        os.remove(CHECKPOINT_PATH)

if cpu_fallback_activated:
    print(">>> CPU fallback was used. Consider upgrading to torch 2.7+ for full SM100 GPU support.")

if "sample_pages_df" in globals() and len(fused_index) != len(sample_pages_df):
    print(f" WARNING: index size ({len(fused_index)}) != page rows ({len(sample_pages_df)})")

print("=" * 60)
print(" BƯỚC 3 HOÀN TẤT (PAGE-LEVEL)")
print(f" Total pages in index: {len(fused_index)}")
print(f" Index path:           {FINAL_INDEX_PATH}")
print("=" * 60)

In [ ]:
# --- BƯỚC 3.5: TRIAL SCORING MODULE (TRAINED HEAD v2) ---
print(">>> BƯỚC 3.5: Defining TrialHead v2...")

import torch
import torch.nn as nn
import torch.nn.functional as F


class TrialHead(nn.Module):
    def __init__(self, embed_dim, lambda_rel=0.5, n_vis_tokens=96):
        super().__init__()
        self.lambda_rel = lambda_rel
        self.embed_dim = embed_dim
        self.n_vis_tokens = n_vis_tokens   # Fix 4: ranh giới image/text

        self.gate_mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim), nn.Mish(),
            nn.Linear(embed_dim, 1)
        )
        self.relation_mlp = nn.Sequential(
            nn.Linear(2 * embed_dim, embed_dim), nn.ReLU(),
            nn.Linear(embed_dim, embed_dim)
        )

        # Fix 8: Gate bias init (chỉ có tác dụng khi train from scratch,
        # weights sẽ bị overwrite khi load_state_dict)
        nn.init.constant_(self.gate_mlp[-1].bias, -1.0)

    def forward(self, q_embs, doc_embs_list):
        B, Nq, D = q_embs.shape
        N = len(doc_embs_list)
        device = q_embs.device

        # Fix 7: L2 normalize cho MaxSim (pure cosine similarity)
        q_norm = F.normalize(q_embs, dim=-1)

        # Fix 1: Sigmoid gate — KHÔNG chia tổng = 1
        gate_logits = self.gate_mlp(q_embs).squeeze(-1)
        q_norms = torch.norm(q_embs.detach(), dim=-1)
        valid_mask = q_norms > 1e-6
        w_q = torch.sigmoid(gate_logits)
        w_q = w_q.masked_fill(~valid_mask, 0.0)

        has_bigrams = (Nq > 1)
        if has_bigrams:
            q_bigram = torch.cat([q_embs[:, 1:], q_embs[:, :-1]], dim=-1)
            q_rel = self.relation_mlp(q_bigram)

        scores = torch.zeros(B, N, device=device)
        for d_idx in range(N):
            d_emb = doc_embs_list[d_idx]
            # Fix 7: L2 normalize doc embeddings
            d_norm = F.normalize(d_emb, dim=-1)

            sim_mat = torch.matmul(q_norm, d_norm.T)
            max_sim, max_j = sim_mat.max(dim=-1)

            rel_contrib = torch.zeros(B, Nq, device=device)
            if has_bigrams:
                d_c = d_emb[max_j[:, 1:].reshape(-1)].reshape(B, Nq - 1, D)
                d_p = d_emb[max_j[:, :-1].reshape(-1)].reshape(B, Nq - 1, D)
                d_rel = self.relation_mlp(torch.cat([d_c, d_p], dim=-1))
                rel_scores = (q_rel * d_rel).sum(dim=-1)

                # Fix 4: Mask cross-modal bigrams
                j_curr = max_j[:, 1:]
                j_prev = max_j[:, :-1]
                same_modality = ((j_curr < self.n_vis_tokens) == (j_prev < self.n_vis_tokens))
                rel_contrib[:, 1:] = rel_scores * same_modality.float()

            token_scores = max_sim + self.lambda_rel * rel_contrib
            scores[:, d_idx] = (w_q * token_scores).sum(dim=-1)

        return scores, w_q

    def get_l1_reg(self, w_q):
        return w_q.abs().mean()


print("✅ TrialHead v2 ready (khớp với train weights)")


In [ ]:
# --- BƯỚC 4: EVALUATION & REPORTING (PAGE-LEVEL, TRIAL Fixed Lambda) ---
print(">>> BƯỚC 4: TRIAL Evaluation & Exporting Reports (Page-Level)...")

import json
import os
import pickle
import torch
import pandas as pd
from tqdm.notebook import tqdm

# ============================================================================== 
# 1. SETUP
# ============================================================================== 
WORKING_DIR = "/kaggle/working"
INDEX_SUFFIX = f"{START_IDX}_{END_IDX}" if "START_IDX" in globals() and "END_IDX" in globals() else "full"
INDEX_PATH = os.path.join(WORKING_DIR, f"colsmol_fused_index_{INDEX_SUFFIX}.pkl")
ANNOTATIONS_PATH = "/kaggle/input/mmdocir-eval-data/MMDocIR_annotations.jsonl"
TRIAL_WEIGHTS_PATH = "/kaggle/input/datasets/namthi/train-weight-0-5/trial_head_weights.pt"

LAMBDA_REL = 0.5

if not os.path.exists(INDEX_PATH):
    raise FileNotFoundError(f"Index file not found: {INDEX_PATH}. Please run BƯỚC 3 first.")
if not os.path.isfile(TRIAL_WEIGHTS_PATH):
    raise FileNotFoundError(f"Trial weights not found: {TRIAL_WEIGHTS_PATH}")

if "fused_index" not in globals() or not fused_index:
    with open(INDEX_PATH, "rb") as f:
        loaded = pickle.load(f)
    fused_index = loaded["fused_index"] if isinstance(loaded, dict) and "fused_index" in loaded else loaded

if "sample_layouts_df" not in globals() or "sample_pages_df" not in globals():
    raise ValueError("sample_layouts_df/sample_pages_df not found. Run BƯỚC 1 first.")
if len(fused_index) == 0:
    raise ValueError("fused_index is empty.")
if len(fused_index) != len(sample_pages_df):
    raise ValueError("fused_index length does not match sample_pages_df length.")

if "page_key_to_index" not in globals():
    page_key_to_index = {k: i for i, k in enumerate(sample_pages_df["page_key"].tolist())}

# Load TrialHead weights
D = fused_index[0].shape[-1]
trial_head = TrialHead(embed_dim=D, lambda_rel=LAMBDA_REL, n_vis_tokens=96).to(device)
trial_head.load_state_dict(torch.load(TRIAL_WEIGHTS_PATH, map_location=device, weights_only=True))
trial_head.eval()
print(f" TrialHead loaded from: {TRIAL_WEIGHTS_PATH}")

# ============================================================================== 
# 2. GENERATE QA PAIRS (PAGE GT)
# ============================================================================== 
print(" Generating QA Pairs (page-level GT)...")

qa_pairs = []
doc_layout_lookup = {k: v.copy() for k, v in sample_layouts_df.groupby("join_doc_name")}
available_docs = set(doc_layout_lookup.keys())
print(f" Docs in RAM: {list(available_docs)}")

matched_docs = 0
with open(ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        try:
            doc_data = json.loads(line)
        except Exception:
            continue

        target_doc = doc_data["doc_name"].replace(".pdf", "")
        if target_doc not in available_docs:
            continue

        matched_docs += 1
        doc_layouts = doc_layout_lookup[target_doc]
        domain = doc_data.get("domain", "General")

        col_page = "page_idx" if "page_idx" in doc_layouts.columns else "page_id"
        doc_layouts["safe_page_eval"] = pd.to_numeric(doc_layouts[col_page], errors="coerce").fillna(-999).astype(int)

        for q_item in doc_data.get("questions", []):
            gt_page_indices = set()

            for target in q_item.get("layout_mapping", []):
                try:
                    t_page = int(target.get("page", -1))
                except Exception:
                    continue

                # Dữ liệu có thể lệch 0-based/1-based, thử cả 2.
                candidate_pages = [t_page, t_page - 1]
                for p in candidate_pages:
                    if p < 0:
                        continue
                    page_key = f"{target_doc}__p{p}"
                    if page_key in page_key_to_index:
                        gt_page_indices.add(page_key_to_index[page_key])

            if gt_page_indices:
                qa_pairs.append(
                    {
                        "question": q_item.get("Q", ""),
                        "gt_indices": sorted(list(gt_page_indices)),
                        "doc_name": target_doc,
                        "domain": domain,
                    }
                )

print(f" Đã quét {matched_docs} tài liệu khớp tên.")
print(f" Found {len(qa_pairs)} QA pairs valid for page-level testing.")

# ============================================================================== 
# 3. SCORING & EXPORT
# ============================================================================== 
if len(qa_pairs) > 0:
    print("\n" + "=" * 60)
    print(f" TRIAL SCORING (PAGE-LEVEL)  lambda = {LAMBDA_REL}")
    print(f" Score(Q,D) = Base_MaxSim + {LAMBDA_REL} x Relation")
    print("=" * 60)

    BATCH_Q = int(PERF_CFG["BATCH_Q_EVAL"])
    DOC_CHUNK_SIZE = int(PERF_CFG["DOC_CHUNK_SIZE"])
    print(f"Using RTX6000 config -> BATCH_Q={BATCH_Q}, DOC_CHUNK_SIZE={DOC_CHUNK_SIZE}")

    all_docs_cpu = [d.to(dtype=torch.float32, device="cpu") for d in fused_index]
    n_docs = len(all_docs_cpu)

    correct_1, correct_5, correct_10 = 0, 0, 0
    total = len(qa_pairs)
    report_data = []

    pbar = tqdm(range(0, total, BATCH_Q), desc="Evaluating")

    for i in pbar:
        batch_qa = qa_pairs[i:i + BATCH_Q]
        queries = [q["question"] for q in batch_qa]

        with torch.no_grad():
            q_inputs = processor.process_queries(queries).to(device)
            q_embeddings = model(**q_inputs).float()

            full_scores = torch.empty((len(batch_qa), n_docs), dtype=torch.float32, device="cpu")
            for d0 in range(0, n_docs, DOC_CHUNK_SIZE):
                d1 = min(d0 + DOC_CHUNK_SIZE, n_docs)
                docs_chunk_gpu = [doc.to(device=device, dtype=torch.float32, non_blocking=True) for doc in all_docs_cpu[d0:d1]]
                chunk_scores, _ = trial_head(q_embeddings, docs_chunk_gpu)
                full_scores[:, d0:d1] = chunk_scores.detach().cpu()

                del docs_chunk_gpu, chunk_scores
                torch.cuda.empty_cache()

        for j, score_row in enumerate(full_scores):
            top_k = min(10, n_docs)
            top_k_indices = torch.topk(score_row, k=top_k).indices.tolist()
            gt_set = set(batch_qa[j]["gt_indices"])

            first_hit = -1
            hits_found = []
            for r, idx in enumerate(top_k_indices):
                if idx in gt_set:
                    if first_hit == -1:
                        first_hit = r + 1
                    hits_found.append(idx)

            if first_hit != -1:
                if first_hit <= 1:
                    correct_1 += 1
                if first_hit <= 5:
                    correct_5 += 1
                if first_hit <= 10:
                    correct_10 += 1

            recall_score = len(hits_found) / len(gt_set) if len(gt_set) > 0 else 0.0
            report_data.append(
                {
                    "lambda_rel": LAMBDA_REL,
                    "doc_name": batch_qa[j]["doc_name"],
                    "domain": batch_qa[j]["domain"],
                    "question": batch_qa[j]["question"],
                    "page_retrieved": str(top_k_indices),
                    "gt_page": str(list(gt_set)),
                    "first_hit_rank": first_hit if first_hit != -1 else "Not Found",
                    "recall": round(recall_score, 2),
                    "is_perfect": recall_score == 1.0,
                }
            )

        if (i + BATCH_Q) % 20 == 0:
            pbar.set_postfix({"R@10": f"{(correct_10 / min(i + BATCH_Q, total)) * 100:.1f}%"})

    print("\n" + "=" * 50)
    print(f" FINAL RESULTS (PAGE-LEVEL, lambda = {LAMBDA_REL})")
    print("=" * 50)
    print(f" Recall@1:  {(correct_1 / total) * 100:.2f}%")
    print(f" Recall@5:  {(correct_5 / total) * 100:.2f}%")
    print(f" Recall@10: {(correct_10 / total) * 100:.2f}%")
    print("=" * 50)

    df_report = pd.DataFrame(report_data)
    report_path = os.path.join(WORKING_DIR, f"trial_report_page_level_lambda_{LAMBDA_REL}.csv")
    df_report.drop(columns=["is_perfect"]).to_csv(report_path, index=False, encoding="utf-8-sig")
    print(f" Report saved: {report_path}")

    err_path = os.path.join(WORKING_DIR, f"trial_error_analysis_page_level_lambda_{LAMBDA_REL}.csv")
    df_report[~df_report["is_perfect"]].drop(columns=["is_perfect"]).to_csv(err_path, index=False)
    print(f" Error analysis saved: {err_path}")

    tokenizer = getattr(processor, "tokenizer", None)
    if tokenizer is None and hasattr(processor, "query_tokenizer"):
        tokenizer = getattr(processor, "query_tokenizer")

    special_ids = set()
    if tokenizer is not None:
        if hasattr(tokenizer, "all_special_ids"):
            try:
                special_ids.update(getattr(tokenizer, "all_special_ids"))
            except Exception:
                pass
        for tid in [
            getattr(tokenizer, "pad_token_id", None),
            getattr(tokenizer, "bos_token_id", None),
            getattr(tokenizer, "eos_token_id", None),
            getattr(tokenizer, "sep_token_id", None),
            getattr(tokenizer, "cls_token_id", None),
            getattr(tokenizer, "unk_token_id", None),
        ]:
            if tid is not None:
                special_ids.add(int(tid))

    def _merge_subword_weights(tokens, weights):
        merged = []
        current_tok = ""
        current_w = 0.0

        for tok, w in zip(tokens, weights):
            if tok is None:
                continue
            if tok.startswith("<") and tok.endswith(">"):
                continue

            if tok.startswith("▁") or tok.startswith("Ġ"):
                if current_tok != "":
                    merged.append((current_tok, current_w))
                current_tok = tok[1:]
                current_w = float(w)
            else:
                if current_tok == "":
                    current_tok = tok
                    current_w = float(w)
                else:
                    current_tok += tok
                    current_w += float(w)

        if current_tok != "":
            merged.append((current_tok, current_w))

        cleaned = []
        for tok, w in merged:
            t = tok.strip()
            if t == "" or t in [",", ".", ":", ";", "-", "_", "(", ")", "[", "]", "{", "}", "'", '"', "/", "?"]:
                continue
            cleaned.append((t, w))

        s = sum(w for _, w in cleaned)
        if s > 1e-12:
            cleaned = [(t, w / s) for t, w in cleaned]
        return cleaned

    N_DIAG = min(5, len(qa_pairs))
    diag_queries = [qa_pairs[i]["question"] for i in range(N_DIAG)]

    print("\n" + "=" * 60)
    print(f" DIAGNOSTIC: Trained Token Weights w_q from TrialHead ({N_DIAG} queries)")
    print("  w_q = normalized gate output learned by 2-MLP TrialHead")
    print("=" * 60)

    diag_doc = all_docs_cpu[0].to(device=device, dtype=torch.float32)
    with torch.no_grad():
        diag_inputs = processor.process_queries(diag_queries).to(device)
        diag_embs = model(**diag_inputs).float()
        _, diag_wq = trial_head(diag_embs, [diag_doc])

    diag_ids = diag_inputs["input_ids"].detach().cpu() if "input_ids" in diag_inputs else None
    diag_wq = diag_wq.detach().cpu()

    for i, query in enumerate(diag_queries):
        if tokenizer is not None and diag_ids is not None:
            ids_i = diag_ids[i].tolist()
            toks = tokenizer.convert_ids_to_tokens(ids_i)
            ws = diag_wq[i].tolist()

            filtered_toks = []
            filtered_ws = []
            for tid, tok, w in zip(ids_i, toks, ws):
                if tid in special_ids:
                    continue
                filtered_toks.append(tok)
                filtered_ws.append(float(w))
        else:
            filtered_toks = [f"tok_{k}" for k in range(len(diag_wq[i]))]
            filtered_ws = [float(x) for x in diag_wq[i].tolist()]

        merged = _merge_subword_weights(filtered_toks, filtered_ws)
        merged.sort(key=lambda x: x[1], reverse=True)

        print(f"\n[Q{i + 1}] {query}")
        print("  " + "-" * 56)
        print(f"  {'Token':<28} {'w_q':>9}   Importance")
        print("  " + "-" * 56)
        for tok, w in merged[:15]:
            bar = "#" * min(int(w * 800), 30)
            print(f"  {tok:<28} {w:>9.5f}   {bar}")

    print("\n" + "=" * 60)
    print(" Note: This is the trained TrialHead weight, not centroid heuristic.")
    print("=" * 60)
else:
    print(" Critical Error: Zero QA Pairs found.")